# LAB·J4 · Eight fake devices

**Hardware:** any machine. ~2 h. No accelerator needed; every exercise runs on one CPU pretending to be eight devices.

A single flag turns one CPU into eight visible JAX devices. Everything this lab does, from meshes to `shard_map`, runs on that fake fleet, on a laptop, the same way it would run on real hardware. The mental model this lab builds, a mesh names your devices, a `PartitionSpec` maps array axes onto that mesh, and the compiler inserts whatever communication your math implies, does not change when the eight fake devices become eight real ones.

One rule matters before anything else: the flag that creates the fake devices has to be set before `jax` is imported, not after. The first code cell below does nothing else.

Predict before you run. Every reveal in this lab follows an empty prediction cell.


In [ ]:
import os

os.environ["XLA_FLAGS"] = "--xla_force_host_platform_device_count=8"

import jax
import jax.numpy as jnp

print(jax.__version__, len(jax.devices()), jax.devices()[0].platform)


## Naming eight devices

A `Mesh` gives your devices names along axes; here, a `4 x 2` grid named `("data", "model")`. A `PartitionSpec` says, for each axis of an array, which mesh axis, if any, splits it. `NamedSharding` is just a mesh and a spec, paired.

Predict what `jax.debug.visualize_array_sharding` draws for an `(8, 16)` array sharded with `P("data", "model")` across that `(4, 2)` mesh: how many tiles, and what shape is each one?


**Your prediction:**


In [ ]:
# run with: XLA_FLAGS=--xla_force_host_platform_device_count=8
import jax
import jax.numpy as jnp
from jax.sharding import NamedSharding, PartitionSpec as P

mesh = jax.make_mesh((4, 2), ("data", "model"))
x = jax.device_put(jnp.ones((8, 16)), NamedSharding(mesh, P("data", "model")))
jax.debug.visualize_array_sharding(x)   # 4x2 tiles, one per device


## Reading the tiles

The visualization draws a grid of eight tiles, four along the array's first axis and two along its second, one tile per device. `P("data", "model")` said: split axis 0 across the mesh's `"data"` axis, four ways, and axis 1 across `"model"`, two ways. Each device now holds a `(2, 8)` slice of the original `(8, 16)` array; put all eight slices back together and you get the array you started with.

Nothing about the array's shape changed from the caller's side. Code written against `x` still sees an `(8, 16)` array; the sharding rides alongside it as metadata, not as a different shape to program against.


## The compiler places your math

Once inputs carry a sharding, `jit` does not compile one program per device by hand. XLA's SPMD partitioner reads the whole jaxpr, works out how sharding propagates from inputs to every intermediate, and inserts whatever communication (all-gather, reduce-scatter, all-reduce) the math actually requires to produce a correctly sharded output. You write ordinary global-array code; the compiler decides what crosses the network between devices.

`jax.lax.with_sharding_constraint` exists for the cases where propagation guesses wrong: it pins one intermediate's layout explicitly, without changing anything else about the program.


## Three ways to shard one matmul

`(8, 16) @ (16, 8)` is small enough to run instantly and simple enough that its communication pattern is worth reading by hand. Below, the same matmul runs three times, each with a different `PartitionSpec` pair for its two operands: splitting the batch axis only (data-style), splitting the shared inner dimension only (model-style), and splitting both.

Predict, for each of the three, whether the lowered program needs a collective operation, a cross-device communication step, to produce a correct answer, or whether each device can compute its slice of the output with no communication at all.


**Your prediction:**


In [ ]:
import jax
import jax.numpy as jnp
from jax.sharding import NamedSharding, PartitionSpec as P

a = jnp.ones((8, 16))
b = jnp.ones((16, 8))

shardings = {
    "data": (P("data", None), P(None, None)),
    "model": (P(None, "model"), P("model", None)),
    "both": (P("data", "model"), P("model", None)),
}

for name, (spec_a, spec_b) in shardings.items():
    sa = jax.device_put(a, NamedSharding(mesh, spec_a))
    sb = jax.device_put(b, NamedSharding(mesh, spec_b))
    text = jax.jit(lambda x, y: x @ y).lower(sa, sb).as_text()
    tags = [t for t in ("all-gather", "all-reduce", "reduce-scatter") if t in text]
    print(f"{name}: {tags or 'no collective tag found'}")


## Checking your prediction against the compiler

The loop prints, for each spec pair, which collective op names, if any, show up in the lowered text. Read the actual strings your run produced rather than trusting a memorized rule: the collective the partitioner reaches for depends on both operands' specs together, not on either one alone, and that dependency is exactly why "predict, then check the lowered text" is a more reliable habit than memorizing a table.

To see more than a yes-or-no tag match, print `text` in full for one of the three cases and read the op names directly; StableHLO spells them out, `all-gather`, `all-reduce`, and their relatives, as ordinary named operations in the program.


## Writing the communication yourself

`jax.shard_map` is the level below automatic placement: it runs your function once per device, on that device's local, already-shrunk block of the array, and any communication across devices has to appear in your code explicitly, as a collective you call by name (`jax.lax.psum`, `ppermute`, and others), addressed by the mesh axis it should cross.

The function below computes a mean over the `"data"` axis using `jax.lax.pmean`. Predict what `global_mean` returns when called on the sharded `(8, 16)` array `x` from earlier: a single scalar, one value per device, or something else.


**Your prediction:**


In [ ]:
from functools import partial

import jax
try:
    from jax import shard_map          # jax 0.6 and later
except ImportError:
    from jax.experimental.shard_map import shard_map   # older releases
from jax.sharding import PartitionSpec as P

@partial(shard_map, mesh=mesh, in_specs=P("data", None), out_specs=P())
def global_mean(block):              # block: this device's (2, 16) slice
    return jax.lax.pmean(block.mean(), axis_name="data")

print(global_mean(x))


## What shard_map actually ran

`out_specs=P()` says the output carries no split axis: the same single value on every device. Each device computed `block.mean()` over only its own local `(2, 16)` slice, a different partial mean per device, and `pmean` then averaged those eight partial means across the `"data"` axis by name, an explicit collective your code called rather than one GSPMD inferred on its own. `global_mean(x)` returns one scalar, identical on every device, because that is what `pmean` guarantees.

Compare this to the matmul section: there, `with_sharding_constraint` and the partitioner did the placing for you, from ordinary global-array code. Here, the shapes inside the function are already local, `(2, 16)`, not `(8, 16)`, and the collective is a line you wrote, not a line the compiler inserted on your behalf. Both levels are correct; they trade automatic inference for explicit control in opposite directions.


## Mark it run

You have sharded one array across eight fake devices, watched the compiler place a matmul's math across a mesh on its own, read the collectives it chose out of the lowered text, and written one yourself with `shard_map`. Go back to chapter 10 on the chapter page and tick LAB·J4 as run.
